# TP – Agentic AI avec LangChain et LangGraph (version corrigée)

Ce TP reprend les thèmes du notebook fourni : configuration, agent simple, modèles dynamiques, mémoire, outils, recherche Web, exécution de code, middlewares, réponses structurées, RAG, protection des données, approbation humaine et LangGraph.

**Utilisation :** exécuter les cellules dans l'ordre. Les étapes nécessitant des services externes sont **optionnelles** et indiquées. Les appels LLM consomment des crédits API. Ne jamais placer les clés dans le notebook. Les outils « météo », « actualités », « employé » et « email » sont des **simulations**, et ne reflètent pas de vraies données/actions.

**Important :** le notebook d'origine contient une clé API exposée. La révoquer et en créer une nouvelle avant toute utilisation.

## 0. Préparer l'environnement

Terminal, à la racine du projet :

```bash
python -m venv .venv
# Windows PowerShell : .venv\Scripts\Activate.ps1
# macOS / Linux : source .venv/bin/activate
python -m pip install -r requirements.txt
```

Copier `.env.example` vers `.env` et y placer votre nouvelle clé API. Ne pas publier `.env`.

Pour utiliser les exemples **OpenAI**, il faut `OPENAI_API_KEY`. Les exemples Tavily nécessitent `TAVILY_API_KEY`. PostgreSQL est optionnel.

In [1]:

import os
from dotenv import load_dotenv
from IPython.display import display, Markdown
from langchain.agents import create_agent
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage

# Load environment variables
load_dotenv(override=True)

# Local Ollama model
MODEL_NAME = "qwen2.5:3b"

# Display the agent's final response
def answer(result):
    display(Markdown(str(result["messages"][-1].content)))

# Initialize local LLM
llm = ChatOllama(
    model=MODEL_NAME,
    temperature=0.1
)

print("Configuration prête ; modèle :", MODEL_NAME)


Configuration prête ; modèle : qwen2.5:3b


## 1. Agent simple et historique des messages

Sans *checkpointer*, deux appels indépendants ne partagent pas automatiquement la conversation. Pour maintenir le contexte, renvoyer les messages précédents ou utiliser la mémoire de la section 3.

In [2]:
simple_agent = create_agent(model=llm, tools=[], system_prompt="You are a helpful assistant.")
first = simple_agent.invoke({"messages": [{"role": "user", "content": "My first name is Omar and my last name is GHILAN."}]})
answer(first)
second = simple_agent.invoke({"messages": first["messages"] + [HumanMessage(content="What is my last name?")]})
answer(second)

Hello Omar GHILAN! It's nice to meet you. How can I assist you today?

Your last name is GHILAN.

## 2. Configuration du modèle et sélection dynamique

Un middleware peut sélectionner un modèle selon le contexte d'exécution (exemple : environnement `test` ou `prod`). Les deux modèles doivent être disponibles sur votre compte.

In [4]:

from typing import TypedDict
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from langchain_ollama import ChatOllama

class EnvironmentContext(TypedDict, total=False):
    env: str

# Local Ollama models
basic_model = ChatOllama(
    model="qwen2.5:3b",
    temperature=0
)

advanced_model = ChatOllama(
    model="qwen2.5:7b",
    temperature=0
)

@wrap_model_call
def select_model(request: ModelRequest, handler) -> ModelResponse:
    env = request.runtime.context.get("env", "test")

    chosen = advanced_model if env == "prod" else basic_model

    print("Modèle sélectionné :", chosen.model)

    return handler(request.override(model=chosen))

dynamic_agent = create_agent(
    model=basic_model,
    tools=[],
    middleware=[select_model],
    context_schema=EnvironmentContext
)

answer(dynamic_agent.invoke(
    {
        "messages": [
            HumanMessage(
                content="Explain what an AI agent is in one sentence."
            )
        ]
    },
    context={"env": "prod"}
))


Modèle sélectionné : qwen2.5:7b


An AI agent is a computer program that acts on behalf of a user or within a system to perform tasks or interactions, often with some degree of autonomy and intelligence.

## 3. Mémoire en session avec LangGraph

`InMemorySaver` conserve les états **pendant la vie du processus Python**, pas après redémarrage. La continuité dépend du `thread_id`.

In [5]:
from langgraph.checkpoint.memory import InMemorySaver
memory_agent = create_agent(model=llm, tools=[], system_prompt="You are helpful.", checkpointer=InMemorySaver())
thread = {"configurable": {"thread_id": "tp-session-1"}}
answer(memory_agent.invoke({"messages": [HumanMessage(content="My name is GHILAN.")]}, config=thread))
answer(memory_agent.invoke({"messages": [HumanMessage(content="What is my name?")]}, config=thread))

That's a great name, GHILAN! How can I assist you today? Are you looking for information about your name, or is there something specific you need help with?

Your name is GHILAN.

### 3.1 Persistance PostgreSQL (optionnel)

Installer `langgraph-checkpoint-postgres` et définir `POSTGRES_URI` dans `.env`. Cette section n'est **pas exécutée par défaut**. Une base PostgreSQL active est nécessaire.

In [8]:

import os
from dotenv import load_dotenv
from langgraph.checkpoint.postgres import PostgresSaver

load_dotenv(override=True)

db_uri = os.environ["POSTGRES_URI"]

with PostgresSaver.from_conn_string(db_uri) as checkpointer:
    checkpointer.setup()
    print("PostgreSQL connected successfully!")
    print("LangGraph checkpoint tables are ready!")

PostgreSQL connected successfully!
LangGraph checkpoint tables are ready!


In [9]:

import os
from dotenv import load_dotenv
from langgraph.checkpoint.postgres import PostgresSaver
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage

load_dotenv(override=True)

db_uri = os.environ["POSTGRES_URI"]

with PostgresSaver.from_conn_string(db_uri) as checkpointer:

    persistent_agent = create_agent(
        model=llm,
        tools=[],
        system_prompt="You are a helpful assistant.",
        checkpointer=checkpointer
    )

    thread = {
        "configurable": {
            "thread_id": "tp-postgres-1"
        }
    }

    result = persistent_agent.invoke(
        {
            "messages": [
                HumanMessage(content="My name is GHILAN.")
            ]
        },
        config=thread
    )

    answer(result)

print("Conversation saved in PostgreSQL!")


Hello GHILAN! It's nice to meet you. How can I assist you today? Do you have any specific questions or topics you'd like to discuss?

Conversation saved in PostgreSQL!


In [ ]:
# Décommenter uniquement si PostgreSQL est configuré.
# from langgraph.checkpoint.postgres import PostgresSaver
# db_uri = os.environ["POSTGRES_URI"]
# with PostgresSaver.from_conn_string(db_uri) as checkpointer:
#     checkpointer.setup()
#     persistent_agent = create_agent(model=llm, tools=[], checkpointer=checkpointer)
#     result = persistent_agent.invoke(
#         {"messages": [HumanMessage(content="Remember my name is Daoudi.")]},
#         config={"configurable": {"thread_id": "persistent-1"}},
#     )
#     answer(result)

## 4. Outils de l'agent

Les outils ci-dessous renvoient **des données simulées**. On les utilise pour montrer comment l'agent choisit et appelle un outil ; ils ne doivent pas être présentés comme des informations en direct.

In [10]:
from langchain_core.tools import tool

@tool
def search_news(query: str) -> str:
    """Return simulated news for a demonstration query."""
    return f"DEMO ONLY — no live news was searched for: {query}"

@tool
def get_weather(location: str) -> str:
    """Return simulated weather, not a real forecast."""
    return f"DEMO ONLY — simulated weather for {location}: sunny, 32°C"

@tool
def get_employee_info(name: str) -> str:
    """Return fictitious employee information for a demonstration."""
    return f"FICTIONAL EMPLOYEE: {name}; role: Developer; salary: 12000 (demo units)"

tools_agent = create_agent(model=llm, tools=[search_news, get_weather, get_employee_info],
                           system_prompt="Use the tools when appropriate. Clearly identify all simulated information as demo data.")
answer(tools_agent.invoke({"messages": [HumanMessage(content="Show the demo weather for Beni Mellal.")]}))

The demo weather for Beni Mellal is sunny with a temperature of 32°C.

In [11]:

from langchain_core.messages import ToolMessage, AIMessage

result = tools_agent.invoke({
    "messages": [
        HumanMessage(content="Show the demo weather for Beni Mellal.")
    ]
})

for message in result["messages"]:
    if isinstance(message, AIMessage) and message.tool_calls:
        print("Tool calls:", message.tool_calls)

    if isinstance(message, ToolMessage):
        print("Tool result:", message.content)

print("Final answer:", result["messages"][-1].content)


Tool calls: [{'name': 'get_weather', 'args': {'location': 'Beni Mellal'}, 'id': 'd0324d33-9fc3-441f-9ca0-9a03e8747427', 'type': 'tool_call'}]
Tool result: DEMO ONLY — simulated weather for Beni Mellal: sunny, 32°C
Final answer: The demo weather for Beni Mellal is sunny with a temperature of 32°C.


## 5. Recherche Web réelle (optionnelle, avec Tavily)

Cette section requiert `TAVILY_API_KEY` ; les résultats dépendent du service. Contrairement aux outils simulés, c'est une recherche externe.

In [12]:
from dotenv import load_dotenv
import os

load_dotenv(override=True)

print("Tavily configured:", bool(os.getenv("TAVILY_API_KEY")))

Tavily configured: True


In [13]:
if os.getenv("TAVILY_API_KEY"):
    from langchain_tavily import TavilySearch
    tavily = TavilySearch(max_results=5, search_depth="basic")
    web_agent = create_agent(model=llm, tools=[tavily],
                             system_prompt="For current information, use web search and include source URLs.")
    answer(web_agent.invoke({"messages": [HumanMessage(content="Find an introductory LangChain agents tutorial and cite the source URL.")]}))
else:
    print("Étape ignorée : définissez TAVILY_API_KEY pour activer la recherche réelle.")

Here are a few introductory LangChain agents tutorials:

1. [Build an AI agent with LangChain in this beginner tutorial](https://www.techtarget.com/news/Build-an-AI-agent-with-LangChain-in-this-beginner-tutorial/834282)
2. [LangChain Agents in 2025 | Full Tutorial for v0.3](https://www.youtube.com/watch?v=Gi7nqB37WEY&vl=en-US)
3. [LangChain Agents for Noobs: A Complete Practical Guide](https://medium.com/@mehulpratapsingh/langchain-agents-for-noobs-a-complete-practical-guide-e231b6c71a4a)
4. [Introduction to LangChain: Build AI Agents with Python](https://academy.langchain.com/courses/foundation-introduction-to-langchain-python)

These resources cover the basics of LangChain installation, creating agents, and using tools such as search and calculators. The tutorials also provide examples of how to manage files on a local computer using LangChain's File System tool. For a more detailed understanding, you might want to explore the full course available on the LangChain Academy.

## 6. Gestion d'erreurs des outils

Un middleware peut transformer une exception d'outil en réponse lisible. On ne révèle pas automatiquement les détails techniques sensibles.

In [14]:
from langchain.agents.middleware import wrap_tool_call
from langchain_core.messages import ToolMessage

@tool
def reciprocal(number: float) -> float:
    """Return reciprocal of a nonzero number."""
    if number == 0:
        raise ValueError("Division by zero")
    return 1 / number

@wrap_tool_call
def handle_tool_errors(request, handler):
    try:
        return handler(request)
    except Exception:
        return ToolMessage(content="Tool failed; check your input and try again.",
                           tool_call_id=request.tool_call["id"])

error_agent = create_agent(model=llm, tools=[reciprocal], middleware=[handle_tool_errors])
answer(error_agent.invoke({"messages": [HumanMessage(content="Calculate the reciprocal of zero using the tool.")]}))

It seems that the reciprocal of zero is undefined because division by zero is not allowed in mathematics. Let's try with a non-zero number instead.

In [15]:

from langchain_core.messages import AIMessage, ToolMessage

result = error_agent.invoke({
    "messages": [
        HumanMessage(
            content="Call the reciprocal tool with number=0 and report its result."
        )
    ]
})

for message in result["messages"]:
    if isinstance(message, AIMessage) and message.tool_calls:
        print("Tool called:", message.tool_calls)

    if isinstance(message, ToolMessage):
        print("Tool response:", message.content)

print("Final answer:", result["messages"][-1].content)


Tool called: [{'name': 'reciprocal', 'args': {'number': 0}, 'id': '2731d99e-04d9-45b1-9686-f00403e3d389', 'type': 'tool_call'}]
Tool response: Tool failed; check your input and try again.
Final answer: It seems that the number you provided is 0. The reciprocal of 0 is undefined because division by zero is not allowed in mathematics. Please provide a different number.


## 7. Prompt système dynamique selon le rôle

In [16]:
from langchain.agents.middleware import dynamic_prompt

class RoleContext(TypedDict, total=False):
    user_role: str

@dynamic_prompt
def role_prompt(request: ModelRequest) -> str:
    role = request.runtime.context.get("user_role", "beginner")
    if role == "expert":
        return "You are a technical AI assistant. Give precise, detailed explanations."
    return "You are a friendly tutor. Explain ideas simply with examples."

role_agent = create_agent(model=llm, tools=[], middleware=[role_prompt], context_schema=RoleContext)
answer(role_agent.invoke({"messages": [HumanMessage(content="What is an AI agent?")]}, context={"user_role": "beginner"}))

An AI agent is like a helper that uses artificial intelligence to make decisions and perform tasks. Think of it as a virtual assistant that can learn and adapt over time, much like how you learn new things as you grow up.

For example, a virtual shopping assistant that recommends products based on your past purchases and browsing history is an AI agent. It learns from your interactions and preferences to suggest items you might like.

Another example could be a virtual coach for exercise, which uses AI to track your workouts, suggest exercises, and provide feedback. It learns your fitness level and preferences to create a personalized training plan.

In both cases, the AI agent is designed to understand and interact with its environment, much like how you interact with the world around you, but it does so using algorithms and data analysis instead of human intuition.

In [17]:
from langchain.agents.middleware import dynamic_prompt

class RoleContext(TypedDict, total=False):
    user_role: str

@dynamic_prompt
def role_prompt(request: ModelRequest) -> str:
    role = request.runtime.context.get("user_role", "beginner")
    if role == "expert":
        return "You are a technical AI assistant. Give precise, detailed explanations."
    return "You are a friendly tutor. Explain ideas simply with examples."

role_agent = create_agent(model=llm, tools=[], middleware=[role_prompt], context_schema=RoleContext)
answer(role_agent.invoke({"messages": [HumanMessage(content="What is an AI agent?")]}, context={"user_role": "expert"}))

An AI agent is a software program or a machine that is designed to perceive its environment, understand its context, and take actions to achieve specific goals or objectives. These agents are capable of learning from their experiences and can interact with their environment in a way that mimics human behavior. AI agents can be found in various applications, including robotics, gaming, automation, and more.

Key characteristics of AI agents include:

1. Perception: They can gather information from their environment through sensors or data inputs.
2. Cognition: They process this information using algorithms and models to understand the context and make decisions.
3. Action: Based on their cognition, they can take actions to achieve their goals or objectives.
4. Learning: Many AI agents are designed to learn from their experiences, allowing them to improve their performance over time.
5. Adaptability: They can adapt to changing environments and new situations.
6. Independence: In many cases, AI agents operate independently of human intervention, though they can also be designed to work in conjunction with humans.

AI agents can be categorized into different types based on their capabilities and applications:

1. Reactive Agents: These agents only consider the current state of their environment and do not have the ability to learn from past experiences. They are often used in simple games or simulations.
2. Limited-History Agents: These agents have some form of memory, allowing them to consider past events, but they still lack the ability to learn from their experiences.
3. Theory of Mind Agents: These agents can understand the mental states of others and can consider the beliefs and intentions of other agents or humans.
4. Self-Aware Agents: These are hypothetical agents that possess self-awareness, a concept that is still being explored in AI research.
5. Autonomous Agents: These agents are capable of operating independently without direct human intervention, often in complex and dynamic environments.

Examples of AI agents include:

- Virtual assistants like Siri, Alexa, or Google Assistant, which use natural language processing to understand and respond to user queries.
- Autonomous vehicles, which use AI to perceive their environment, make decisions, and navigate roads.
- Chatbots, which simulate human conversation to provide customer service or engage in discussions.
- Robots, which can be programmed to perform specific tasks or interact with their environment in various ways.
- Recommendation systems, which use AI to suggest products or content based on user preferences and behavior.
- Game-playing agents, such as those used in chess or Go, which use algorithms to analyze game states and make optimal moves.

In summary, AI agents are sophisticated software programs designed to perform tasks autonomously, often in complex and dynamic environments. They are essential components in many modern applications and are continually advancing in capabilities and applications.

## 8. Réponse structurée avec Pydantic

In [19]:
from pydantic import BaseModel, Field
from langchain.agents.structured_output import ToolStrategy

class ContactInfo(BaseModel):
    name: str = Field(description="Contact full name")
    email: str = Field(description="Email address")
    phone: str = Field(description="Phone number")

structured_agent = create_agent(model=llm, tools=[], response_format=ToolStrategy(ContactInfo))
result = structured_agent.invoke({"messages": [HumanMessage(content="Extract contact: Omar GHILAN, ghilanomar818@gmail.com +212706394945") ]})
print(result["structured_response"].model_dump())

{'name': 'Omar GHILAN', 'email': 'ghilanomar818@gmail.com', 'phone': '+212706394945'}


## 9. Middleware personnalisé pour le suivi d'exécution

In [20]:
from langchain.agents.middleware import AgentMiddleware, AgentState

class HooksDemo(AgentMiddleware):
    def before_agent(self, state: AgentState, runtime):
        print("before_agent")
        return None
    def before_model(self, state: AgentState, runtime):
        print("before_model")
        return None
    def after_model(self, state: AgentState, runtime):
        print("after_model")
        return None
    def after_agent(self, state: AgentState, runtime):
        print("after_agent")
        return None

hooks_agent = create_agent(model=llm, tools=[], middleware=[HooksDemo()])
answer(hooks_agent.invoke({"messages": [HumanMessage(content="Name one benefit of LangChain.")]}))

before_agent
before_model
after_model
after_agent


One benefit of LangChain is that it provides a comprehensive framework for integrating large language models with databases, APIs, and other data sources. This integration allows for more efficient and effective data processing and analysis, enabling users to leverage the capabilities of large language models to derive insights from a wide range of data sources.

## 10. RAG avec FAISS : outil de recherche documentaire

Le mini-corpus fourni ci-dessous est **fictif à but pédagogique**. Les embeddings nécessitent un accès API. La recherche ne doit répondre qu'à partir des passages récupérés.

In [23]:

from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.tools import create_retriever_tool
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage
from langchain.agents import create_agent

# 1. Local models
advanced_model = ChatOllama(
    model="qwen2.5:7b",
    temperature=0
)

embeddings = OllamaEmbeddings(
    model="nomic-embed-text"
)

# 2. Knowledge base
texts = [
    "Le TP présente les principes des agents IA avec LangChain.",
    "LangGraph peut conserver l'état d'une conversation avec un checkpointer.",
    "La recherche RAG utilise une base vectorielle pour retrouver des documents pertinents.",
    "La validation humaine peut précéder les actions sensibles de l'agent.",
]

# 3. Create FAISS vector database
vector_store = FAISS.from_texts(
    texts,
    embedding=embeddings
)

# 4. Create retriever
retriever = vector_store.as_retriever(
    search_kwargs={"k": 3}
)

# 5. Create retrieval tool
retrieval_tool = create_retriever_tool(
    retriever,
    name="tp_knowledge_search",
    description="Search the provided TP knowledge base for facts."
)

# 6. Create RAG Agent
rag_agent = create_agent(
    model=advanced_model,
    tools=[retrieval_tool],
    system_prompt=(
        "Always call tp_knowledge_search before answering. "
        "Use only information from retrieved passages. "
        "If the passages do not contain the answer, say so. "
        "Answer in French."
    )
)

# 7. Ask a question
result = rag_agent.invoke({
    "messages": [
        HumanMessage(
            content="Comment fonctionne la mémoire dans ce TP ?"
        )
    ]
})

# 8. Inspect tool execution
for message in result["messages"]:

    if isinstance(message, AIMessage) and message.tool_calls:
        print("Tool called:", message.tool_calls)

    if isinstance(message, ToolMessage):
        print("Retrieved documents:", message.content)

# 9. Final response
print("Final answer:", result["messages"][-1].content)


Tool called: [{'name': 'tp_knowledge_search', 'args': {'query': 'mémoire dans ce TP'}, 'id': 'a83c0bfe-056e-4860-b941-63cfe493c862', 'type': 'tool_call'}]
Retrieved documents: Le TP présente les principes des agents IA avec LangChain.

LangGraph peut conserver l'état d'une conversation avec un checkpointer.

La validation humaine peut précéder les actions sensibles de l'agent.
Final answer: Le TP ne fournit pas de détails spécifiques sur la mémoire. Cependant, il mentionne que LangGraph peut conserver l'état d'une conversation avec un checkpointer, ce qui implique une forme de mémoire pour l'agent IA. La validation humaine peut également précéder les actions sensibles de l'agent, suggérant une gestion de l'état ou d'un contexte.


## 11. Garde-fous et données personnelles (PII)

La configuration suivante masque les adresses e-mail dans le texte transmis au modèle. Elle ne remplace pas les contrôles de sécurité et de confidentialité réels.

In [ ]:
from langchain.agents.middleware import PIIMiddleware

pii_agent = create_agent(model=llm, tools=[], middleware=[PIIMiddleware("email", strategy="redact", apply_to_input=True)])
answer(pii_agent.invoke({"messages": [HumanMessage(content="Repeat the email you see: ghilanomar818@gmail.com")]}))

I see that you've provided an email address that has been redacted. However, I don't have access to any specific email content as it wasn't provided in my initial context. If you'd like me to assist with something related to emails or have a specific question about email content, please provide more details or context.

## 12. Human-in-the-loop : approbation avant un « envoi » simulé

Le nom de l'outil est `draft_email_demo` : **aucun e-mail réel n'est envoyé**. Le premier appel s'arrête avant l'outil ; la décision explicite permet de poursuivre. Ne pas automatiser l'approbation pour de vraies actions sensibles.

In [25]:
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.types import Command

@tool
def draft_email_demo(recipient: str, subject: str) -> str:
    """Simulate an email action; never actually send anything."""
    return f"DEMO APPROVED (NOT SENT): to={recipient}, subject={subject}"

approval_agent = create_agent(
    model=llm,
    tools=[draft_email_demo],
    middleware=[HumanInTheLoopMiddleware(interrupt_on={"draft_email_demo": True})],
    checkpointer=InMemorySaver(),
    system_prompt="When asked to simulate sending an email, use draft_email_demo. Explain it is only a simulation.",
)
approval_config = {"configurable": {"thread_id": "approval-demo-1"}}
pending = approval_agent.invoke({"messages": [HumanMessage(content="Simulate sending an email to demo@example.com with subject TP Agentic.")]}, config=approval_config)
print("Interruption / demande d'approbation :", pending.get("__interrupt__", "aucune interruption"))

Interruption / demande d'approbation : [Interrupt(value={'action_requests': [{'name': 'draft_email_demo', 'args': {'recipient': 'demo@example.com', 'subject': 'TP Agentic'}, 'description': "Tool execution requires approval\n\nTool: draft_email_demo\nArgs: {'recipient': 'demo@example.com', 'subject': 'TP Agentic'}"}], 'review_configs': [{'action_name': 'draft_email_demo', 'allowed_decisions': ['approve', 'edit', 'reject', 'respond']}]}, id='915ee604fcbc6febc87ff13dfd616118', response_schema=None)]


In [26]:

approved = approval_agent.invoke(Command(resume={"decisions": [{"type": "approve"}]}), config=approval_config)
answer(approved)

The email has been simulated and is set to be sent to demo@example.com with the subject "TP Agentic". However, please note that this is a simulation and the email will not actually be sent.

## 13. Garde-fou avant l'agent (exemple simplifié)

Filtrage pédagogique d'un mot-clef ; une liste de mots n'est pas une solution de sécurité complète.

In [27]:
from langchain.agents.middleware import before_agent

@before_agent(can_jump_to=["end"])
def simple_content_filter(state: AgentState, runtime):
    latest_user = next((m for m in reversed(state["messages"]) if m.type == "human"), None)
    if latest_user and isinstance(latest_user.content, str) and "forbidden-demo-token" in latest_user.content.lower():
        return {"messages": [{"role": "assistant", "content": "Cette requête de démonstration est bloquée."}], "jump_to": "end"}
    return None

filtered_agent = create_agent(model=llm, tools=[], middleware=[simple_content_filter])
answer(filtered_agent.invoke({"messages": [HumanMessage(content="Test forbidden-demo-token")]}))

Cette requête de démonstration est bloquée.

## 14. LangGraph : graphe minimal sans API externe

Cette dernière section complète l'en-tête « LangGraph » laissé vide dans le notebook initial. Le graphe ci-dessous est déterministe : il n'utilise pas de modèle payant.

In [28]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

class GraphState(TypedDict):
    question: str
    response: str

def respond(state: GraphState) -> dict:
    return {"response": f"Le graphe a reçu : {state['question']}"}

graph_builder = StateGraph(GraphState)
graph_builder.add_node("respond", respond)
graph_builder.add_edge(START, "respond")
graph_builder.add_edge("respond", END)
minimal_graph = graph_builder.compile()
print(minimal_graph.invoke({"question": "Qu'est-ce qu'un agent ?", "response": ""}))

{'question': "Qu'est-ce qu'un agent ?", 'response': "Le graphe a reçu : Qu'est-ce qu'un agent ?"}


## 15. Bilan et questions de révision

1. Quelle est la différence entre l'historique transmis explicitement et `InMemorySaver` ?
2. À quel moment un agent utilise-t-il un outil ?
3. Pourquoi un outil de météo simulée ne suffit-il pas pour une prévision réelle ?
4. Quelle différence entre RAG et mémoire de conversation ?
5. Pourquoi l'approbation humaine est-elle essentielle avant un outil sensible ?
6. Pourquoi faut-il stocker les clés API dans `.env` plutôt que dans le notebook ?

**Limites de validation :** les cellules ont été vérifiées syntaxiquement, mais les appels aux API externes et PostgreSQL doivent être testés avec vos propres accès. Les modèles, forfaits et versions de bibliothèques peuvent modifier la compatibilité.


## 15. Bilan et questions de révision — Réponses

### 1. Quelle est la différence entre l'historique transmis explicitement et `InMemorySaver` ?

L'historique explicite consiste à transmettre manuellement les messages précédents à chaque nouvel appel du modèle. En revanche, `InMemorySaver` permet à LangGraph de sauvegarder automatiquement l'état de la conversation et de le retrouver grâce à un `thread_id`. Cependant, cette mémoire est temporaire et disparaît lorsque le processus Python est arrêté.

### 2. À quel moment un agent utilise-t-il un outil ?

Un agent utilise un outil lorsque le modèle estime qu'une fonction externe est nécessaire ou utile pour répondre à la demande de l'utilisateur. Par exemple, dans ce TP, l'agent utilise `get_weather` pour obtenir une météo simulée et `tp_knowledge_search` pour rechercher des informations dans une base documentaire.

### 3. Pourquoi un outil de météo simulée ne suffit-il pas pour une prévision réelle ?

Un outil de météo simulée retourne des informations fictives définies directement dans le code. Il ne consulte aucune source météorologique réelle. Pour obtenir une prévision fiable, il faut utiliser un service météo fournissant des données actualisées.

### 4. Quelle différence entre RAG et mémoire de conversation ?

Le RAG (Retrieval-Augmented Generation) permet de rechercher des informations pertinentes dans une base documentaire externe, par exemple avec FAISS et des embeddings.

La mémoire conversationnelle permet de conserver l'historique et l'état des échanges entre l'utilisateur et l'agent, par exemple avec `InMemorySaver` ou `PostgresSaver`.

Ainsi, le RAG apporte des connaissances externes, tandis que la mémoire conserve le contexte de la conversation.

### 5. Pourquoi l'approbation humaine est-elle essentielle avant un outil sensible ?

L'approbation humaine permet de contrôler les actions sensibles avant leur exécution, comme l'envoi d'un email ou la modification de données.

Dans ce TP, `HumanInTheLoopMiddleware` interrompt l'exécution de l'agent et attend une décision humaine avant d'autoriser l'utilisation de l'outil.

Cela permet de réduire les risques d'erreurs et de garder le contrôle sur les actions importantes.

### 6. Pourquoi faut-il stocker les clés API dans `.env` plutôt que dans le notebook ?

Les clés API sont des informations confidentielles. Les écrire directement dans un notebook risque de les exposer lors du partage du projet ou de sa publication sur GitHub.

Le fichier `.env` permet de séparer les informations sensibles du code source. Il doit être exclu du suivi Git à l'aide de `.gitignore`.

### Conclusion

Ce TP nous a permis de découvrir les principaux concepts des agents IA avec LangChain et LangGraph : création d'agents, gestion de la mémoire, utilisation d'outils, recherche RAG, middleware, gestion des erreurs, sorties structurées et validation humaine.

Les expérimentations ont été réalisées principalement avec des modèles locaux via Ollama, notamment Qwen2.5, afin de limiter la dépendance aux API payantes.

L'utilisation de PostgreSQL a également permis de tester le stockage des conversations, même si la persistance après redémarrage n'a pas été vérifiée expérimentalement.
